In [1]:
from skimage.transform import resize
from skimage.io import imread, imshow
from PIL import Image
import os, os.path
import random
from skimage.feature import hog
from sklearn.svm import SVC
import numpy as np
import time
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import cv2
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
data = []
categories = ['bmp', 'btr', 'cars', 'grad', 'howitzer', 'tank']
for i in categories:
    path = 'C:\\Users\\Admin\\Desktop\\1991\\Machine Learning\\HW\\4\\train\\' + i
    valid_images = [".jpg",".gif",".png",".tga"]
    for f in os.listdir(path):
        data.append([np.array(imread(os.path.join(path,f))), i])

random.shuffle(data)
X_train = []
y_train = []
for i in range(len(data)):
    X_train.append(data[i][0])
    y_train.append(data[i][1])        
        
X_train = np.array(X_train)        
y_train = np.array(y_train)

In [3]:
y_val = []
imgs = []
categories = ['bmp', 'btr', 'car', 'grad', 'howitzer', 'tank']
for i in categories:
    path = 'C:\\Users\\Admin\\Desktop\\1991\\Machine Learning\\HW\\4\\val\\' + i
    valid_images = [".jpg",".gif",".png",".tga"]
    for f in os.listdir(path):
        imgs.append(np.array(imread(os.path.join(path,f))))
        y_val.append(i)
X_val = np.array(imgs)

In [4]:
y_test = []
imgs = []
categories = ['bmp', 'btr', 'cars', 'grad', 'howitzer', 'tank']
for i in categories:
    path = 'C:\\Users\\Admin\\Desktop\\1991\\Machine Learning\\HW\\4\\test_images\\' + i
    valid_images = [".jpg",".gif",".png",".tga"]
    for f in os.listdir(path):
        imgs.append(np.array(imread(os.path.join(path,f))))
        y_test.append(i)
X_test = np.array(imgs)

In [5]:
# This is data augmentaion functions

def fill(img, h, w):
    img = cv2.resize(img, (h, w), cv2.INTER_CUBIC)
    return img
        
def horizontal_shift(img, ratio=0.0):
    if ratio > 1 or ratio < 0:
        print('Value should be less than 1 and greater than 0')
        return img
    ratio = np.random.uniform(-ratio, ratio)
    h, w = img.shape[:2]
    to_shift = w*ratio
    if ratio > 0:
        img = img[:, :int(w-to_shift), :]
    if ratio < 0:
        img = img[:, int(-1*to_shift):, :]
    img = fill(img, h, w)
    return img




def vertical_shift(img, ratio=0.0):
    if ratio > 1 or ratio < 0:
        print('Value should be less than 1 and greater than 0')f
        return img
    ratio = np.random.uniform(-ratio, ratio)
    h, w = img.shape[:2]
    to_shift = h*ratio
    if ratio > 0:
        img = img[:int(h-to_shift), :, :]
    if ratio < 0:
        img = img[int(-1*to_shift):, :, :]
    img = fill(img, h, w)
    return img



def brightness(img, low, high):
    value = np.random.uniform(low, high)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hsv = np.array(hsv, dtype = np.float64)
    hsv[:,:,1] = hsv[:,:,1]*value
    hsv[:,:,1][hsv[:,:,1]>255]  = 255
    hsv[:,:,2] = hsv[:,:,2]*value 
    hsv[:,:,2][hsv[:,:,2]>255]  = 255
    hsv = np.array(hsv, dtype = np.uint8)
    img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return img

def rotation(img, angle):
    angle = int(random.uniform(-angle, angle))
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((int(w/2), int(h/2)), angle, 1)
    img = cv2.warpAffine(img, M, (w, h))
    return img


def noisy(image):
    row,col,ch= image.shape
    mean = 0
    var = 0.1
    sigma = var**0.5
    gauss = np.random.normal(mean,sigma,(row,col,ch))
    gauss = gauss.reshape(row,col,ch)
    noisy = image + gauss
    return noisy  

In [6]:
def data_augment(X_train, y_train, n_times):
    X_train = list(X_train)
    y_train = list(y_train)
    for _ in range(n_times):
        for i in range(len(X_train)):
            a = np.random.uniform(0,0.4)
            angle = np.random.uniform(10,360)
            X_train.append(horizontal_shift(X_train[i], 0.35))
            y_train.append(y_train[i])

            X_train.append(vertical_shift(X_train[i], 0.35))
            y_train.append(y_train[i])

            
            
            
            
        
        
            X_train.append(brightness(X_train[i], 0.5, 2))
            y_train.append(y_train[i])



            X_train.append(cv2.blur(X_train[i], ksize = (3,3)))
            y_train.append(y_train[i])
            
            X_train.append(cv2.blur(X_train[i], ksize = (3,3)))
            y_train.append(y_train[i])
            
        
            
            
            
            X_train.append(cv2.flip(X_train[i], flipCode = 1))
            y_train.append(y_train[i])
            
            X_train.append(rotation(X_train[i], angle))
            y_train.append(y_train[i])
            
    
    return np.array(X_train, dtype = 'int'), np.array(y_train)
    
    
    
X_train_augmented, y_train_augmented = data_augment(X_train, y_train, 1)    

In [7]:
#flattening 
def flatten(X):
    X_train_flat = []
    for i in X:
        X_train_flat.append(i.ravel())
    X_train_flat = np.array(X_train_flat)
    
    return X_train_flat

X_train_flat = flatten(X_train)
X_train_augmented_flat = flatten(X_train_augmented)
X_val_flat = flatten(X_val)
X_test_flat = flatten(X_test)

In [8]:
# this is hog implementation

def  hog_iamge(X_train, X_test):
    X_train_hog = []
    for i in X_train:

        resized_img = resize(i, (128,64)) 

        fd, hog_image = hog(resized_img, orientations=9, pixels_per_cell=(8, 8), 
                            cells_per_block=(2, 2), visualize=True, multichannel=True)

        X_train_hog.append(hog_image)
    X_train_hog = np.array(X_train_hog)

    X_test_hog = []
    for i in X_test:

        resized_img = resize(i, (128,64)) 

        fd, hog_image = hog(resized_img, orientations=9, pixels_per_cell=(8, 8), 
                            cells_per_block=(2, 2), visualize=True, multichannel=True)

        X_test_hog.append(hog_image)
    X_test_hog = np.array(X_test_hog)


    X_test_hog_flat = []
    X_train_hog_flat = []
    for i in X_test_hog:
        X_test_hog_flat.append(i.ravel())

    for i in X_train_hog:
        X_train_hog_flat.append(i.ravel())

    X_test_hog_flat = np.array(X_test_hog_flat)
    X_train_hog_flat = np.array(X_train_hog_flat)

    X_test_hog_flat = []
    X_train_hog_flat = []
    for i in X_test_hog:
        X_test_hog_flat.append(i.ravel())

    for i in X_train_hog:
        X_train_hog_flat.append(i.ravel())

    X_test_hog_flat = np.array(X_test_hog_flat)
    X_train_hog_flat = np.array(X_train_hog_flat)
    return X_train_hog_flat, X_test_hog_flat

X_train_hog_flat, X_val_hog_flat  = hog_iamge(X_train_augmented, X_val)
X_train_hog_flat, X_test_hog_flat  = hog_iamge(X_train_augmented, X_test)




In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_hog_flat_norm = scaler.fit_transform(X_train_hog_flat)
X_val_hog_flat_norm = scaler.fit_transform(X_val_hog_flat)
X_test_hog_flat_norm = scaler.fit_transform(X_test_hog_flat)

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_flat_norm = scaler.fit_transform(X_train_flat)
X_val_flat_norm = scaler.fit_transform(X_val_flat)
X_test_flat_norm = scaler.fit_transform(X_test_flat)

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_augmented_flat_norm = scaler.fit_transform(X_train_augmented_flat)


In [24]:
# this is with hog which didn't work well

start = time.perf_counter()
classifier = SVC(C=0.01, kernel='linear', gamma=1)
classifier.fit(X_train_hog_flat_norm, y_train_augmented)
print(f'Time taken: {time.perf_counter()-start} sec')
print('Train accuracy: ', accuracy_score(classifier.predict(X_train_hog_flat_norm), y_train_augmented))
print('Valid accuracy: ', accuracy_score(classifier.predict(X_val_hog_flat_norm), y_val))
print('Test accuracy: ', accuracy_score(classifier.predict(X_test_hog_flat_norm), y_test))

Time taken: 2.6042813999999908 sec
Train accuracy:  1.0
Valid accuracy:  0.3
Test accuracy:  0.3630952380952381


In [27]:
#This is without data augmentaion 

start = time.perf_counter()
classifier = SVC(C=0.01, kernel='linear')
classifier.fit(X_train_flat_norm, y_train)
print(f'Time taken: {time.perf_counter()-start} sec')
print('Train accuracy: ', accuracy_score(classifier.predict(X_train_flat_norm), y_train))
print('Val accuracy: ', accuracy_score(classifier.predict(X_val_flat_norm), y_val))
print('Test accuracy: ', accuracy_score(classifier.predict(X_test_flat_norm), y_test))

Time taken: 1.4934985000000438 sec
Train accuracy:  1.0
Val accuracy:  0.36666666666666664
Test accuracy:  0.31547619047619047


In [28]:
#This is the model with data augmentaion

start = time.perf_counter()
classifier = SVC(C=1, kernel='poly', degree=3, gamma = 1.5)
classifier.fit(X_train_augmented_flat_norm, y_train_augmented)
print(f'Time taken: {time.perf_counter()-start} sec')
print('Train accuracy: ', accuracy_score(classifier.predict(X_train_augmented_flat_norm), y_train_augmented))
print('Val accuracy: ', accuracy_score(classifier.predict(X_val_flat_norm), y_val))
print('Test accuracy: ', accuracy_score(classifier.predict(X_test_flat_norm), y_test))

Time taken: 81.2887219999999 sec
Train accuracy:  1.0
Val accuracy:  0.2
Test accuracy:  0.26785714285714285


In [153]:
start = time.perf_counter()
from sklearn.model_selection import GridSearchCV
parameters = {'C': [0.01 ],
              'kernel': ['poly', 'linear'],
              'degree' : [2, 3]
                }

model = SVC()
clf = GridSearchCV(model, parameters,
                   scoring='accuracy', cv=3)
clf.fit(X_train_augmented_flat , y_train_augmented)
print(time.perf_counter()-start)

166.63868889999958


In [157]:
clf.best_params_
# clf.best_score_
# clasification_repor
# clf.return_train_score

{'C': 0.01, 'kernel': 'linear'}

In [12]:
#This is Deep leraning model with data augmentaion

y_train_enc = []
categories = ['bmp', 'btr', 'cars', 'grad', 'howitzer', 'tank']
for i in y_train_augmented:
    y_train_enc.append(categories.index(i))
    
y_train_enc = np.array(y_train_enc)


y_val_enc = []
categories = ['bmp', 'btr', 'car', 'grad', 'howitzer', 'tank']
for i in y_val:
    y_val_enc.append(categories.index(i))
    
y_val_enc = np.array(y_val_enc)

y_test_enc = []
categories = ['bmp', 'btr', 'cars', 'grad', 'howitzer', 'tank']
for i in y_test:
    y_test_enc.append(categories.index(i))
    
y_test_enc = np.array(y_test_enc)




#without flattening input
from tensorflow import keras
import tensorflow as tf
model = keras.Sequential([
#     keras.layers.Flatten(input_shape = (480, )),
    keras.layers.Dense(100, activation = "relu", input_shape = (196608,)),
    keras.layers.Dropout(0.2),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(6, activation = "softmax")
])


model.compile(
    optimizer = keras.optimizers.Adam(0.01) ,
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

In [37]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense_2 (Dense)              (None, 100)               19660900  
_________________________________________________________________
dropout (Dropout)            (None, 100)               0         
_________________________________________________________________
batch_normalization_1 (Batch (None, 100)               400       
_________________________________________________________________
dense_3 (Dense)              (None, 6)                 606       
Total params: 19,661,906
Trainable params: 19,661,706
Non-trainable params: 200
_________________________________________________________________


In [15]:
model.fit(X_train_augmented_flat_norm , y_train_enc, epochs=5)

Epoch 1/5
15/15 [==============================] - 2s 132ms/step - loss: 1.0591 - accuracy: 0.6083
Epoch 2/5
15/15 [==============================] - 2s 129ms/step - loss: 1.0005 - accuracy: 0.6354
Epoch 3/5
15/15 [==============================] - 2s 128ms/step - loss: 0.9176 - accuracy: 0.6812
Epoch 4/5
15/15 [==============================] - 2s 144ms/step - loss: 0.8563 - accuracy: 0.7000
Epoch 5/5
15/15 [==============================] - 2s 160ms/step - loss: 0.7816 - accuracy: 0.7333


In [14]:
model.save('my_model_svm')

INFO:tensorflow:Assets written to: my_model_svm\assets


In [19]:
print(f'Validation accuracy : {(np.argmax(model.predict(X_val_flat_norm), axis = 1) == y_val_enc).sum() / len(X_val)}')
print(f'Testb accuracy : {(np.argmax(model.predict(X_test_flat_norm), axis = 1) == y_test_enc).sum() / len(X_test)}')

Validation accuracy : 0.3333333333333333
Testb accuracy : 0.3333333333333333
